# 04, Priority weight and sentiment calibration

**Inputs:** none external, v1 has no historical priority-labelled ticket log, so this notebook
generates a synthetic labelled sample from a held-out "true" weight vector plus label noise, to
demonstrate and validate the calibration methodology end to end.

**Outputs:** `config/priority_weights.yaml` (weights only, the domain policy in
`libs/domain/policy/priority.py` stays code; ADR-010).

The domain policy stays code; only the six weights are data. This notebook calibrates the
*weights*, not the scoring formula itself.

In [1]:
import os, sys
from pathlib import Path

# Notebook hygiene: always run from the v1 root, never hard-code absolute d:\... paths.
_here = Path.cwd()
_v1_root = _here if (_here / "pyproject.toml").exists() else _here.parents[0]
os.chdir(_v1_root)
sys.path.insert(0, str(_v1_root))

# Notebooks that touch libs.platform.registry/config must pin a profile explicitly, the
# process env has no APP_PROFILE by default, which silently falls back to "cpu" (real Ollama
# calls, no stub) and produces confusing failures/latency. Pin stub here; override the env var
# before importing libs.platform.config in the rare notebook that needs otherwise.
os.environ.setdefault("APP_PROFILE", "stub")
print("cwd:", Path.cwd(), "| APP_PROFILE:", os.environ["APP_PROFILE"])

cwd: d:\DSEP22\v1 | APP_PROFILE: stub


In [2]:
import random

import yaml
from sklearn.metrics import cohen_kappa_score

from libs.domain.policy.priority import DEFAULT_WEIGHTS, TriageInputs, score

random.seed(22)
SIGNAL_KEYS = ["sentiment", "urgency_keywords", "customer_segment", "outage_scope", "prior_contacts", "sla_age"]

c:\Users\Bimsara\miniconda3\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_version" in AudioTranscript has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\Bimsara\miniconda3\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_version" in VisualSummary has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\Bimsara\miniconda3\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_version" in Provenance has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


## Lexicon-based sentiment baseline

`triage_svc` currently classifies sentiment via a small keyword lexicon (angry/frustrated
markers). Evaluate it against a small hand-labelled sample of representative ticket openers.

In [3]:
SENTIMENT_SAMPLES = [
    ("This is absolutely unacceptable, I've called three times about the same issue.", "angry"),
    ("I am furious that no one has fixed this yet.", "angry"),
    ("This is ridiculous, fix it now.", "angry"),
    ("I'm still not getting internet, this is the third time this month.", "frustrated"),
    ("Again my connection dropped, I'm getting annoyed.", "frustrated"),
    ("I've tried the reset again and it's still not working.", "frustrated"),
    ("Hi, could you help me check my internet connection please?", "neutral"),
    ("My router light is red, can you assist?", "neutral"),
    ("Just wanted to ask about upgrading my plan.", "neutral"),
    ("Thanks so much, the technician did a great job yesterday!", "satisfied"),
    ("Really happy with how quickly this got resolved.", "satisfied"),
    ("Great service, appreciate the quick fix.", "satisfied"),
]

_ANGRY_MARKERS = ("angry", "furious", "unacceptable", "ridiculous", "worst", "outrageous")
_FRUSTRATED_MARKERS = ("frustrated", "again", "still not", "third time", "annoyed", "fed up")


def lexicon_sentiment(text: str) -> str:
    lowered = text.lower()
    if any(m in lowered for m in _ANGRY_MARKERS):
        return "angry"
    if any(m in lowered for m in _FRUSTRATED_MARKERS):
        return "frustrated"
    return "neutral"  # the lexicon has no positive-sentiment markers -> "satisfied" is unreachable


preds = [lexicon_sentiment(text) for text, _ in SENTIMENT_SAMPLES]
labels = [label for _, label in SENTIMENT_SAMPLES]
accuracy = sum(p == l for p, l in zip(preds, labels)) / len(labels)
print(f"lexicon sentiment accuracy: {accuracy:.2f}")
list(zip(labels, preds))

lexicon sentiment accuracy: 0.75


[('angry', 'angry'),
 ('angry', 'angry'),
 ('angry', 'angry'),
 ('frustrated', 'frustrated'),
 ('frustrated', 'frustrated'),
 ('frustrated', 'frustrated'),
 ('neutral', 'neutral'),
 ('neutral', 'neutral'),
 ('neutral', 'neutral'),
 ('satisfied', 'neutral'),
 ('satisfied', 'neutral'),
 ('satisfied', 'neutral')]

Result: the lexicon catches angry/frustrated/neutral but has no positive marker list, so
"satisfied" is structurally unreachable, a real limitation to flag in `06-vlm-and-asr.md`'s
sibling doc, `08-evaluation.md`. Fixing it only needs a small positive-marker list; left as a
follow-up rather than scope creep here.

## Synthetic labelled priority sample

Generate 200 tickets with random signal values, label them with a held-out "true" weight vector
plus 8% label noise (simulating human disagreement at band boundaries), then treat those labels
as ground truth for calibration, exactly the workflow a real historical-ticket sample would
follow once v1 has been in production long enough to accumulate one.

In [4]:
TRUE_WEIGHTS = {
    "sentiment": 30, "urgency_keywords": 15, "customer_segment": 10,
    "outage_scope": 25, "prior_contacts": 5, "sla_age": 15,
}
BAND_ORDER = ["critical", "high", "normal", "low"]


def gen_ticket() -> TriageInputs:
    return {
        "sentiment_score": round(random.random(), 2),
        "urgency_keyword_hit": random.random() < 0.3,
        "customer_segment_score": random.choice([0.3, 1.0]),
        "outage_scope_score": round(random.random(), 2) if random.random() < 0.2 else 0.0,
        "prior_contacts_score": round(random.random(), 2),
        "sla_age_score": round(random.random(), 2),
    }


def true_band(inputs: TriageInputs, noise: float = 0.08) -> str:
    _, band, _ = score(inputs, TRUE_WEIGHTS)
    if random.random() < noise:
        idx = BAND_ORDER.index(band.value)
        idx = max(0, min(len(BAND_ORDER) - 1, idx + random.choice([-1, 1])))
        return BAND_ORDER[idx]
    return band.value


tickets = [gen_ticket() for _ in range(200)]
labels = [true_band(t) for t in tickets]
from collections import Counter
Counter(labels)

Counter({'normal': 102, 'low': 79, 'high': 18, 'critical': 1})

## Randomised grid search over the weight simplex

A full 6-dimensional integer grid summing to 100 has tens of thousands of points; a full sweep
isn't necessary to demonstrate the calibration methodology, so this samples the simplex (Dirichlet
draw scaled to sum 100) and keeps the best-scoring point. Swap for `itertools.product` over a
coarse per-weight grid if an exhaustive search is ever needed.

In [5]:
default_preds = [score(t, DEFAULT_WEIGHTS)[1].value for t in tickets]
default_kappa = cohen_kappa_score(labels, default_preds)
print("default weights kappa:", round(default_kappa, 4))

best_kappa, best_weights = None, None
for _trial in range(800):
    raw = [random.random() for _ in SIGNAL_KEYS]
    total = sum(raw)
    candidate = {k: round(v / total * 100) for k, v in zip(SIGNAL_KEYS, raw)}
    candidate[SIGNAL_KEYS[0]] += 100 - sum(candidate.values())  # fix rounding drift back to 100

    preds = [score(t, candidate)[1].value for t in tickets]
    kappa = cohen_kappa_score(labels, preds)
    if best_kappa is None or kappa > best_kappa:
        best_kappa, best_weights = kappa, candidate

print("calibrated weights kappa:", round(best_kappa, 4))
best_weights

default weights kappa: 0.646
calibrated weights kappa: 0.7037


{'sentiment': 22,
 'urgency_keywords': 15,
 'customer_segment': 8,
 'outage_scope': 25,
 'prior_contacts': 12,
 'sla_age': 18}

## Save the calibrated weights

In [6]:
out_path = Path("config/priority_weights.yaml")
out_path.write_text(
    yaml.safe_dump(
        {
            "weights": best_weights,
            "calibration": {
                "method": "randomised simplex search, 800 trials, Cohen's kappa vs synthetic labels",
                "kappa": round(best_kappa, 4),
                "default_weights_kappa": round(default_kappa, 4),
                "sample_size": len(tickets),
            },
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)
print(f"wrote {out_path}")

RepresenterError: ('cannot represent an object', np.float64(0.7037))

In [ ]:
import json

with open("evaluation/reports/metrics.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps({
        "notebook": "04_priority_and_sentiment",
        "sentiment_lexicon_accuracy": accuracy,
        "default_weights_kappa": default_kappa,
        "calibrated_weights_kappa": best_kappa,
    }) + "\n")
print("appended metrics row")

appended metrics row
